In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Chaotic Associative Memory Cryptography\n",
    "## Security Analysis\n",
    "\n",
    "Statistical evaluation of ciphertext produced by CAMC.\n",
    "Metrics: uniform byte distribution, high Shannon entropy,\n",
    "avalanche effect near 50%, and near-zero autocorrelation."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from network import ChaoticOscillatoryNetwork\n",
    "from crypto import CAMCCipher\n",
    "from dynamics import ciphertext_histogram, entropy, bit_flip_analysis, autocorrelation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.rcParams.update({\n",
    "    'figure.facecolor': '#fafafa',\n",
    "    'axes.facecolor': '#fafafa',\n",
    "    'axes.edgecolor': '#222222',\n",
    "    'axes.labelcolor': '#222222',\n",
    "    'text.color': '#222222',\n",
    "    'font.size': 11,\n",
    "    'axes.grid': True,\n",
    "    'grid.alpha': 0.3,\n",
    "    'grid.color': '#999999'\n",
    "})"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "N_NEURONS = 128\n",
    "WEIGHT_SCALE = 2.0\n",
    "PLAINTEXT_LONG = \"A\" * 1000\n",
    "PLAINTEXT_SHORT = \"Security evaluation message for CAMC cipher.\"\n",
    "\n",
    "np.random.seed(42)\n",
    "network = ChaoticOscillatoryNetwork(n_neurons=N_NEURONS, weight_scale=WEIGHT_SCALE)\n",
    "sync_state = np.exp(1j * np.random.uniform(0, 2 * np.pi, N_NEURONS))\n",
    "cipher = CAMCCipher(network, sync_state)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Byte Histogram"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plain_bytes = np.frombuffer(PLAINTEXT_LONG.encode('utf-8'), dtype=np.uint8)\n",
    "encrypted = cipher.encrypt(PLAINTEXT_LONG)\n",
    "cipher_bytes = np.frombuffer(encrypted, dtype=np.uint8)\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "axes[0].hist(plain_bytes, bins=256, color='#2d4a2b', alpha=0.9)\n",
    "axes[0].set_title('Plaintext Byte Distribution')\n",
    "axes[0].set_xlabel('Byte value')\n",
    "axes[0].set_ylabel('Count')\n",
    "axes[1].hist(cipher_bytes, bins=256, color='#4a7c59', alpha=0.9)\n",
    "axes[1].set_title('Ciphertext Byte Distribution')\n",
    "axes[1].set_xlabel('Byte value')\n",
    "axes[1].set_ylabel('Count')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Shannon Entropy"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plain_entropy = entropy(PLAINTEXT_LONG.encode('utf-8'))\n",
    "cipher_entropy = entropy(encrypted)\n",
    "print(f'Plaintext entropy : {plain_entropy:.4f} bits/byte')\n",
    "print(f'Ciphertext entropy: {cipher_entropy:.4f} bits/byte')\n",
    "print(f'Max possible      : 8.0000 bits/byte')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Avalanche Effect\n",
    "Change one bit in plaintext, measure output bit difference."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "mean_avalanche, std_avalanche = bit_flip_analysis(cipher, PLAINTEXT_SHORT, trials=200)\n",
    "print(f'Avalanche effect: {mean_avalanche*100:.2f}% ± {std_avalanche*100:.2f}%')\n",
    "print('Ideal: 50%')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Autocorrelation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ac = autocorrelation(encrypted, max_lag=50)\n",
    "plt.figure(figsize=(10, 3))\n",
    "plt.stem(range(len(ac)), ac)\n",
    "plt.title('Ciphertext Autocorrelation')\n",
    "plt.xlabel('Lag (bytes)')\n",
    "plt.ylabel('Correlation coefficient')\n",
    "plt.axhline(0, color='#999999', linewidth=0.5)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Summary\n",
    "\n",
    "Measured metrics confirm the keystream exhibits\n",
    "high entropy, uniform distribution, near-optimal avalanche,\n",
    "and negligible autocorrelation."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}